# GIADA IV-C1 v3 — prospective actual-solver-clock confirmation
The v1/v2 schedules and forensic trace are diagnostic only. This notebook uses disjoint v3 events, unchanged canonical mechanisms and numeric gates. A pass is limited to IV-C1; IV-C2, IV-C3 and Task33 remain separate.


In [ ]:
from pathlib import Path
import json,subprocess,sys,uuid
from IPython.display import display
WORK=Path('/kaggle/working')/('giada_iv_c1_v3_'+uuid.uuid4().hex[:8]);WORK.mkdir()
REPO=WORK/'giada';TEACHER=WORK/'neuron_as_deep_net'
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER)],check=True)
subprocess.run(['git','-C',str(TEACHER),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
try:
 import neuron
except ImportError:
 subprocess.run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7'],check=True)
 import neuron
assert neuron.__version__.split('+')[0]=='8.2.7'
print({'revision':REVISION,'neuron':neuron.__version__})


In [ ]:
OUTPUT=Path('/kaggle/working/artifacts')/('giada_iv_c1_v3_'+REVISION[:7]+'_'+uuid.uuid4().hex[:8])
completed=subprocess.run([sys.executable,'-u',str(REPO/'scripts/run_iv_c1_deterministic_synapse.py'),'--teacher',str(TEACHER),'--output',str(OUTPUT),'--contract','v3'],check=False)
path=OUTPUT/('failure_report.json' if (OUTPUT/'failure_report.json').exists() else 'final_report.json')
report=json.loads(path.read_text()) if path.is_file() else {'valid':False,'error':f'worker interrupted: {completed.returncode}'}
display({k:report.get(k) for k in ('valid','iv_c1_passed','selected_phase_steps','calibration_scores_us','negative_controls','iv_c2_passed','iv_c3_passed','task33_authorized','error')})
if report.get('confirmation'):
 display({k:{m:round(v[m],10) for m in ('max_g_error_us','max_state_error','max_i_error_na','max_charge_error_na_ms','max_holding_error_mv','passed')} for k,v in report['confirmation'].items()})
print({'output':str(OUTPUT),'exit':completed.returncode})
